# MiniMax Music 3 — Google Colab 音樂產生

先在 Colab 選擇 `執行階段 → 變更執行階段類型 → GPU`，然後點選「全部執行」。上方**模型準備**儲存格會自動安裝套件、檢查資源並載入模型；它可摺疊，通常在一個執行階段只需執行一次。下方**修改 Prompt 並產生**儲存格可反覆編輯、單獨執行，不會重新下載或載入模型。

如要將權重快取在 Google Drive，先展開模型準備儲存格，把 `USE_DRIVE_CACHE` 改成 `True`。首次仍需下載數十 GB，之後的 Colab 執行階段可重用 Drive 快取；Drive 掛載需授權與足夠空間，讀取速度也可能較慢。建議使用支援 BF16 的 L4/A100；低於 24 GiB 顯存還需至少 32 GiB 系統 RAM。常見的 T4 不支援此 BF16 流程。

本 notebook 依據 [Diffusers 0.40.0 MiniMax Music 3 文件](https://huggingface.co/docs/diffusers/v0.40.0/en/api/pipelines/minimax_music3)。輸出為 44.1 kHz 立體聲 WAV。

In [ ]:
#@title 1. 模型準備（首次執行；可摺疊） { display-mode: "form" }
USE_DRIVE_CACHE = False  # 改成 True，可把模型權重快取在 Google Drive

import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError('請先在 Colab 選擇 GPU 執行階段，再重新執行。')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('此 GPU 不支援 BF16；請選擇 L4 或 A100 等較新 GPU。')
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print(f'GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB | RAM: {ram_gib:.1f} GiB')
if vram_gib < 24 and ram_gib < 32:
    raise RuntimeError('低顯存模式需要至少 32 GiB 系統 RAM；請啟用高 RAM 執行階段。')

if 'pipe' in globals():
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    model_cache_dir = Path('/content/drive/MyDrive/MiniMax-Music3/hub-cache')
    model_cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'
else:
    if shutil.disk_usage('/content').free < 35 * 2**30:
        raise RuntimeError('Colab 暫存磁碟不足 35 GiB；請清理空間或改用 Google Drive 快取。')
    model_cache_dir = Path('/content/minimax-music3-hub-cache')
    model_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ['HF_HUB_CACHE'] = str(model_cache_dir)
print(f'模型快取：{model_cache_dir}')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.40.0', 'transformers', 'accelerate', 'soundfile', 'opencc',
], check=True)

from diffusers import ComponentsManager, ModularPipeline
from diffusers.hooks.group_offloading import apply_group_offloading

model_id = 'MiniMaxAI/MiniMax-Music3'
cache_kwargs = {'cache_dir': str(model_cache_dir)}
if vram_gib >= 24:
    pipe = ModularPipeline.from_pretrained(model_id, **cache_kwargs)
    pipe.load_components(dtype=torch.bfloat16, **cache_kwargs)
    pipe.to('cuda')
else:
    manager = ComponentsManager()
    manager.enable_auto_cpu_offload(device='cuda')
    pipe = ModularPipeline.from_pretrained(model_id, components_manager=manager, **cache_kwargs)
    pipe.load_components(dtype=torch.bfloat16, **cache_kwargs)
    apply_group_offloading(
        pipe.language_model,
        onload_device=torch.device('cuda'),
        offload_type='leaf_level',
        use_stream=True,
    )
print('模型已載入。接下來只需重跑下方的產生儲存格。')

## 修改 Prompt 並產生音樂

編輯下方儲存格開頭的 `lyrics` 和 `music_description`，再執行此儲存格。每次重跑會沿用已載入的模型，並另存一個 WAV；想要同樣描述的不同版本，請改變 `seed`。段落標籤（例如 `[Verse]`、`[Chorus]`）須獨立一行。`duration_seconds` 是最長秒數，模型可能提早結束。送入模型前，兩個輸入中的中文都會以 OpenCC `t2s` 轉成簡體字。

In [ ]:
# 只要修改以下四個設定，然後重跑此儲存格；不需重跑上方的模型準備。
lyrics = """[Verse]
Morning light filtering through the pine
Every quiet street is yours and mine
[Chorus]
Softly the world begins to breathe"""
music_description = """A warm acoustic pop song at 96 BPM with intimate female vocals,
fingerpicked guitar, soft piano, brushed drums, and a gradual build into a wide final chorus."""
duration_seconds = 10  # 1–300 秒
seed = 7  # 改變 seed 可產生另一個版本

if 'pipe' not in globals():
    raise RuntimeError('請先執行上方的「模型準備」儲存格。')
if not lyrics.strip() or not music_description.strip():
    raise ValueError('歌詞與曲風描述都不可留空。')
if not 1 <= duration_seconds <= 300 or seed < 0:
    raise ValueError('秒數須為 1–300，seed 須為非負整數。')

from pathlib import Path
import soundfile as sf
from opencc import OpenCC
from IPython.display import Audio, display
from google.colab import files

converter = OpenCC('t2s.json')
model_lyrics = converter.convert(lyrics)
model_description = converter.convert(music_description)
print('送入模型的歌詞：\n' + model_lyrics)
print('送入模型的音樂描述：\n' + model_description)

audio = pipe(
    prompt=model_description,
    lyrics=model_lyrics,
    audio_duration=float(duration_seconds),
    generator=torch.Generator('cuda').manual_seed(seed),
    output='audios',
)[0]
audio_array = audio.T.float().cpu().numpy() if isinstance(audio, torch.Tensor) else audio.T
output_dir = Path('/content/minimax_music3_outputs')
output_dir.mkdir(exist_ok=True)
take_number = globals().get('music3_take_number', 0) + 1
music3_take_number = take_number
output_path = output_dir / f'minimax_music3_take{take_number:02d}_seed{seed}.wav'
sf.write(output_path, audio_array, pipe.sampling_rate, subtype='PCM_16')
info = sf.info(output_path)
assert info.channels == 2 and info.frames > 0, '輸出的 WAV 無效或不是立體聲。'
print(f'已儲存：{output_path} | {info.samplerate} Hz | {info.channels} 聲道 | {info.duration:.1f} 秒')
display(Audio(filename=str(output_path)))
files.download(str(output_path))